# REINFORCE on CartPole

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

REINFORCE is the simplest policy-gradient method: roll out an episode, score each action by the discounted return that followed it, and bump the log-probability of good actions up.


## Mathematical Formulation

$$\nabla_\theta J(\theta) = \mathbb{E}\!\left[\sum_{t} \nabla_\theta \log \pi_\theta(a_t \mid s_t)\,G_t\right],\quad G_t = \sum_{k \geq t} \gamma^{k-t} r_k$$

Subtract a baseline (mean return or value function) to reduce variance.


## Implementation


In [ ]:
# pip install gymnasium
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class Policy(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, 64), nn.ReLU(),
                                 nn.Linear(64, n_actions))
    def forward(self, obs):
        return self.net(obs)

def discount(rewards, gamma=0.99):
    g = 0
    out = []
    for r in reversed(rewards):
        g = r + gamma * g
        out.insert(0, g)
    return torch.tensor(out)


## Experiment


In [ ]:
env = gym.make('CartPole-v1')
policy = Policy(env.observation_space.shape[0], env.action_space.n)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

returns_log = []
for ep in range(150):
    obs, _ = env.reset(seed=ep)
    log_probs, rewards = [], []
    done = False
    while not done:
        obs_t = torch.tensor(obs, dtype=torch.float32)
        logits = policy(obs_t)
        dist = torch.distributions.Categorical(logits=logits)
        a = dist.sample()
        log_probs.append(dist.log_prob(a))
        obs, r, term, trunc, _ = env.step(a.item())
        rewards.append(r)
        done = term or trunc
    G = discount(rewards)
    G = (G - G.mean()) / (G.std() + 1e-8)  # baseline
    loss = -torch.stack(log_probs) @ G
    opt.zero_grad(); loss.backward(); opt.step()
    returns_log.append(sum(rewards))
    if ep % 25 == 0:
        avg = sum(returns_log[-25:]) / max(1, min(25, len(returns_log)))
        print(f'episode {ep:3d}  avg return (25) {avg:.1f}')


## Discussion

- High variance is the main weakness — baselines and actor-critic help.
- The policy is on-policy: every gradient comes from data generated by the *current* policy.
- For continuous action spaces use a Gaussian policy and sample with `Normal`.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
